# Professional Practice X4: Semi-Supervised Learning

## Leveraging Unlabeled Data with Label Propagation

**Production Challenge**: You have 10,000 images but can only afford to label 100. Can you still build a good classifier?

**Answer**: YES! Semi-supervised learning leverages unlabeled data to improve performance.

**Goal**: Compare supervised vs. semi-supervised learning with varying amounts of labeled data.

**Key Technique**: Label Propagation - spread labels through the data graph.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.semi_supervised import LabelPropagation, LabelSpreading
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix
import seaborn as sns

np.random.seed(42)
print('✅ Loaded!')

## Step 1: Load and Prepare Data

We'll simulate a low-label scenario.

In [ ]:
# Load digits
digits = load_digits()
X = digits.data / 16.0  # Normalize
y = digits.target

print(f"Dataset: {X.shape}")
print(f"Samples: {len(X)}")
print(f"Features: {X.shape[1]}")
print(f"Classes: {len(np.unique(y))}")

# Split into train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=42)

print(f"\nTrain: {len(X_train)}, Test: {len(X_test)}")

# Visualize samples
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.flatten()):
    ax.imshow(X_train[i].reshape(8, 8), cmap='gray')
    ax.set_title(f'Label: {y_train[i]}')
    ax.axis('off')
plt.suptitle('Sample Digits from Training Set', fontweight='bold')
plt.tight_layout()
plt.show()
print('✅ Data loaded!')

## Step 2: Create Low-Label Scenarios

Simulate different labeling budgets: 1%, 5%, 10%, 20%.

In [ ]:
# Test different label percentages
label_percentages = [0.01, 0.05, 0.10, 0.20, 0.50, 1.0]

print("=== LABELING SCENARIOS ===\n")
for pct in label_percentages:
    n_labeled = int(pct * len(X_train))
    print(f"{pct*100:>5.1f}% labeled: {n_labeled:>4} samples labeled, {len(X_train) - n_labeled:>4} unlabeled")

print('\n✅ Scenarios defined!')

## Step 3: Implement Semi-Supervised Learning

Use Label Propagation to spread labels to unlabeled data.

In [ ]:
def evaluate_semi_supervised(X_train, y_train, X_test, y_test, label_percentage):
    """Train and evaluate semi-supervised model"""
    # Create partially labeled dataset
    n_labeled = int(label_percentage * len(X_train))
    
    # Copy labels and mark unlabeled as -1
    y_train_partial = y_train.copy()
    unlabeled_indices = np.arange(n_labeled, len(y_train))
    y_train_partial[unlabeled_indices] = -1
    
    # Train Label Propagation
    lp = LabelPropagation(kernel='rbf', gamma=20, max_iter=1000)
    lp.fit(X_train, y_train_partial)
    
    # Predict
    y_pred = lp.predict(X_test)
    acc_semi = accuracy_score(y_test, y_pred)
    
    # Supervised baseline (only on labeled data)
    clf = LogisticRegression(max_iter=1000, random_state=42)
    clf.fit(X_train[:n_labeled], y_train[:n_labeled])
    y_pred_supervised = clf.predict(X_test)
    acc_supervised = accuracy_score(y_test, y_pred_supervised)
    
    return acc_semi, acc_supervised, n_labeled

# Run experiments
results = []
for pct in label_percentages:
    acc_semi, acc_supervised, n_labeled = evaluate_semi_supervised(
        X_train, y_train, X_test, y_test, pct
    )
    results.append({
        'percentage': pct,
        'n_labeled': n_labeled,
        'semi_supervised': acc_semi,
        'supervised': acc_supervised,
        'improvement': acc_semi - acc_supervised
    })
    print(f"Labels: {pct*100:>5.1f}% ({n_labeled:>4} samples)")
    print(f"  Semi-supervised: {acc_semi:.4f}")
    print(f"  Supervised only: {acc_supervised:.4f}")
    print(f"  Improvement: +{(acc_semi - acc_supervised):.4f}")
    print()

print('✅ Experiments complete!')

## Step 4: Visualize Results

In [ ]:
# Extract results
import pandas as pd
df_results = pd.DataFrame(results)

# Plot accuracy comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy curves
ax = axes[0]
ax.plot(df_results['percentage'] * 100, df_results['semi_supervised'], 
       'bo-', linewidth=2, markersize=8, label='Semi-Supervised')
ax.plot(df_results['percentage'] * 100, df_results['supervised'], 
       'ro-', linewidth=2, markersize=8, label='Supervised')
ax.set_xlabel('Percentage of Labeled Data', fontsize=12)
ax.set_ylabel('Test Accuracy', fontsize=12)
ax.set_title('Semi-Supervised vs Supervised Learning', fontweight='bold', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 100)

# Improvement
ax = axes[1]
bars = ax.bar(df_results['percentage'] * 100, df_results['improvement'] * 100,
             color='green', alpha=0.7, edgecolor='black', linewidth=1.5)
ax.axhline(y=0, color='black', linestyle='-', linewidth=1)
ax.set_xlabel('Percentage of Labeled Data', fontsize=12)
ax.set_ylabel('Improvement (%)', fontsize=12)
ax.set_title('Semi-Supervised Improvement', fontweight='bold', fontsize=13)
ax.grid(True, alpha=0.3, axis='y')

# Add value labels
for bar, val in zip(bars, df_results['improvement'] * 100):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
           f'+{val:.1f}%' if val > 0 else f'{val:.1f}%',
           ha='center', va='bottom' if val > 0 else 'top', 
           fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()
print('✅ Visualization complete!')

## Step 5: Detailed Analysis at 5% Labels

Deep dive into what Label Propagation learns.

In [ ]:
# Train with 5% labels
label_pct = 0.05
n_labeled = int(label_pct * len(X_train))

y_train_partial = y_train.copy()
y_train_partial[n_labeled:] = -1

# Train
lp = LabelPropagation(kernel='rbf', gamma=20, max_iter=1000)
lp.fit(X_train, y_train_partial)

# Get predicted labels for unlabeled data
y_unlabeled_pred = lp.transduction_[n_labeled:]
y_unlabeled_true = y_train[n_labeled:]

# Accuracy on unlabeled data
unlabeled_acc = accuracy_score(y_unlabeled_true, y_unlabeled_pred)

print(f"=== LABEL PROPAGATION ANALYSIS (5% labeled) ===\n")
print(f"Labeled samples: {n_labeled}")
print(f"Unlabeled samples: {len(y_unlabeled_true)}")
print(f"\nLabel propagation accuracy on unlabeled data: {unlabeled_acc:.4f}")
print(f"This means {unlabeled_acc*100:.1f}% of pseudo-labels are correct!")

# Test accuracy
y_pred_test = lp.predict(X_test)
test_acc = accuracy_score(y_test, y_pred_test)
print(f"\nTest accuracy: {test_acc:.4f}")
print('\n✅ Analysis complete!')

## Step 6: Confusion Matrix

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred_test)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
           xticklabels=range(10), yticklabels=range(10),
           cbar_kws={'label': 'Count'})
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.title(f'Confusion Matrix (5% labeled, Accuracy: {test_acc:.4f})', 
         fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()
print('✅ Confusion matrix complete!')

## Step 7: Compare Label Propagation vs Label Spreading

In [ ]:
# Compare algorithms
label_pct = 0.10
n_labeled = int(label_pct * len(X_train))
y_train_partial = y_train.copy()
y_train_partial[n_labeled:] = -1

# Label Propagation (hard clamping)
lp = LabelPropagation(kernel='rbf', gamma=20)
lp.fit(X_train, y_train_partial)
acc_lp = lp.score(X_test, y_test)

# Label Spreading (soft clamping)
ls = LabelSpreading(kernel='rbf', gamma=20, alpha=0.2)
ls.fit(X_train, y_train_partial)
acc_ls = ls.score(X_test, y_test)

# Supervised baseline
clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(X_train[:n_labeled], y_train[:n_labeled])
acc_supervised = clf.score(X_test, y_test)

print(f"=== ALGORITHM COMPARISON (10% labeled) ===\n")
print(f"Supervised (Logistic Regression): {acc_supervised:.4f}")
print(f"Label Propagation: {acc_lp:.4f} (+{acc_lp - acc_supervised:.4f})")
print(f"Label Spreading: {acc_ls:.4f} (+{acc_ls - acc_supervised:.4f})")

# Visualize
methods = ['Supervised\nOnly', 'Label\nPropagation', 'Label\nSpreading']
accuracies = [acc_supervised, acc_lp, acc_ls]
colors = ['red', 'blue', 'green']

plt.figure(figsize=(10, 6))
bars = plt.bar(methods, accuracies, color=colors, alpha=0.7, 
              edgecolor='black', linewidths=1.5)
plt.ylabel('Test Accuracy', fontsize=12)
plt.title('Semi-Supervised Algorithm Comparison (10% labeled)', 
         fontweight='bold', fontsize=14)
plt.ylim(0, 1.0)
plt.grid(True, alpha=0.3, axis='y')

# Add value labels
for bar, acc in zip(bars, accuracies):
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
            f'{acc:.4f}',
            ha='center', va='bottom', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()
print('\n✅ Comparison complete!')

## Production Best Practices

### When to Use Semi-Supervised Learning?

✅ **Perfect for**:
- Limited labeling budget
- Expensive labeling (medical, legal, expert knowledge)
- Large unlabeled datasets available
- Data with natural cluster structure

❌ **Avoid when**:
- Labeled and unlabeled data from different distributions
- Very small total dataset (< 1000 samples)
- Unlabeled data is noisy or corrupted

### Algorithm Comparison

| Method | Best For | Key Parameter |
|--------|----------|--------------|
| **Label Propagation** | Well-separated clusters | gamma (RBF width) |
| **Label Spreading** | Noisy labels | alpha (clamping factor) |
| **Self-Training** | Any classifier | confidence threshold |
| **Co-Training** | Multiple feature views | diversity |

### Production Pipeline

```python
from sklearn.semi_supervised import LabelPropagation
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV

# 1. Prepare data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Mark unlabeled
y_partial = y.copy()
y_partial[n_labeled:] = -1

# 2. Tune hyperparameters
param_grid = {'gamma': [10, 20, 30]}
lp = LabelPropagation(kernel='rbf')
grid = GridSearchCV(lp, param_grid, cv=3)
grid.fit(X_scaled[:n_labeled], y[:n_labeled])

# 3. Train on all data (labeled + unlabeled)
best_model = grid.best_estimator_
best_model.fit(X_scaled, y_partial)

# 4. Use pseudo-labels for supervised learning
y_pseudo = best_model.transduction_
clf = LogisticRegression()
clf.fit(X_scaled, y_pseudo)
```

### Key Insights from Results

1. **Biggest gains at low label percentages**: 5-10% labeled data benefits most
2. **Diminishing returns**: Above 50% labels, semi-supervised ≈ supervised
3. **Quality of unlabeled data matters**: Clean unlabeled data → better propagation
4. **Hyperparameter tuning is critical**: gamma controls neighborhood size

### Checklist

✅ **Validate unlabeled data quality** (remove outliers)  
✅ **Tune gamma/alpha** parameters carefully  
✅ **Start with 5-10%** labeled data for best ROI  
✅ **Compare to supervised baseline** to measure improvement  
✅ **Use cross-validation** on labeled data  
✅ **Monitor pseudo-label confidence** (transduction_)  
✅ **Consider active learning** to select most informative samples to label

### Real-World Applications

- **Medical Imaging**: Few labeled scans, many unlabeled
- **Text Classification**: Limited expert annotations
- **Fraud Detection**: Few confirmed fraud cases
- **Speech Recognition**: Transcription is expensive
- **Sentiment Analysis**: Manual labeling is time-consuming